# B2-024 — Session 4: Semi-Supervised Learning by Pseudo-Labeling

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

Labels are expensive and unlabelled data is cheap.
In the `shapes_ssl` task you have 30 labelled training images, 30 labelled validation images, a pool of 600 unlabelled images, and a locked test set of 300.
This session teaches two ways to use the pool: confidence-thresholded self-training (a model labels the pool for its own retraining) and cluster-then-label (`book1:k-means` on frozen features, with clusters named by the labelled points).
It also teaches when these methods hurt: confirmation bias, class imbalance, and leakage through validation.
The model throughout is Book 1's small CNN (`book1:convolution`, `book1:cnn-training`) trained with `book1:softmax` and `book1:cross-entropy-loss`.

## 1. The limited-label regime and the supervised baseline

**The images.** Each `shapes_ssl` image is $8\times8$ grey pixels in $[0,1]$ showing one of three shapes, a square frame (class 0), a plus (class 1), or a diagonal cross (class 2), with random size, stroke width, contrast, a sub-pixel position jitter, and heavy pixel noise, clipped to $[0,1]$.
The noise is strong enough that a single image can be ambiguous.

**The model.** `ShapeCNN`: `Conv2d(1, 8, 3)` → ReLU → `Conv2d(8, 16, 3)` → ReLU → flatten ($16\cdot4\cdot4=256$) → `Linear(256, 3)`, giving logits; `F.cross_entropy` applies the softmax internally.
It has $80+1168+771=2019$ parameters, far more than 30 examples can pin down, so it **overfits** (`book1:overfitting`): the training loss reaches nearly zero while held-out accuracy stays well below what more labels would give.

**The supervised baseline** is the reference every semi-supervised method must beat under the same protocol.
In this unit it is fixed: `torch.manual_seed(SEED)`, construct `ShapeCNN`, Adam with learning rate $0.01$, 150 full-batch steps on the 30 labelled training images.
`capstone_data.baseline_score("shapes_ssl")` computes its locked-test accuracy with this committed recipe ($0.8100$ at the frozen seed), and any learner who follows the recipe reproduces the same model exactly.

**Validation comes from labelled data only.** The 30 validation images are labelled and are never added to training, never pseudo-labelled, and never used to fit clusters.
They are the only data on which you choose a threshold, a number of rounds, or between methods.
With 30 validation images each one is worth $1/30\approx0.033$ of accuracy, so validation can separate large effects but not small ones; that limitation is why the semi-supervised recipes in this unit are pinned and only a few choices are left to validation.

**Checkpoint 1A.** How many parameters does `Linear(256, 3)` have, and what fraction of ShapeCNN's parameters is that?

**Checkpoint 1B.** Why may the 30 validation images not be added to the training set "since they are labelled anyway"?

**Collected answers.** $256\cdot3+3=771$, about $38\%$ of $2019$. Because they are the only unbiased basis for every choice you make; once trained on, their accuracy is optimistic and no data is left for selection (the test set must stay locked).

## 2. Confidence-threshold self-training

**Self-training** alternates two steps for a fixed number of rounds:

1. **Pseudo-label.** The current model (the *teacher*) predicts class probabilities $p_i=\operatorname{softmax}(f(x_i))$ for every pool image. Its predicted label is $\hat y_i=\arg\max_k p_{ik}$ and its **confidence** is $c_i=\max_k p_{ik}$. Keep only images with $c_i\ge\tau$ (a threshold such as $0.9$), subject to the class-balance rule of Section 3.
2. **Retrain.** Train a *student* on the labelled images plus the selected pool images with their pseudo-labels. The student becomes the next round's teacher.

Round 0's teacher is the supervised baseline.
In this unit every student is **re-initialized** from the same seed (rather than continuing from the teacher's weights), so each round is an independent fit to the current labelled-plus-pseudo-labelled set, and the round count is a clean knob.

**What the threshold does.** A high $\tau$ admits few pool images, but they are more likely correct; a low $\tau$ admits more, with more errors.
Because a model trained on 30 images is typically overconfident, many pool images reach confidence near $1$, and the threshold alone filters less than you would hope; the per-class cap and the noise of Section 4 do much of the work.

**Selection on validation.** Candidate thresholds (for example $\{0.7, 0.9\}$) are compared by the final round's accuracy on the labelled validation images, with a pre-registered tie rule (prefer the larger threshold, which admits fewer errors).

**Cost.** Each round is one full training run.
Three rounds of 150 steps on at most $30+600$ images take a few seconds on a CPU; the accelerator extension of p17 runs the same loop with a larger pool, more rounds, and a wider CNN.

**Checkpoint 2A.** A pool image has probabilities $(0.05, 0.91, 0.04)$ and $\tau=0.9$. What pseudo-label and confidence does it get, and is it kept?

**Checkpoint 2B.** Why re-initialize the student each round instead of continuing to train the teacher?

**Collected answers.** Label 1, confidence 0.91; kept (subject to the class cap), since $0.91\ge0.9$. So each round is an independent fit to its own training set: continuing from the teacher carries over its overfitted solution and makes round-to-round comparisons depend on accumulated optimizer history.

## 3. Class balance and deterministic selection

A teacher that prefers one class labels most of the pool as that class, and self-training then amplifies the preference: the student sees more of that class, predicts it more, and the next round admits even more.
`book1:class-imbalance` showed how a skewed training set skews a classifier; pseudo-labeling can *create* the skew.

**Per-class caps.** Keep at most $m$ pseudo-labels per predicted class.
Within each class, rank the candidates (those with $c_i\ge\tau$ and $\hat y_i=k$) by confidence, highest first, and keep the top $m$.
Caps can grow across rounds ($100$, then $200$, then $200$ per class in this unit), so early rounds add only the most confident images.

**Determinism.** Selection must be a function of the probabilities alone, so it is reproducible and testable:

* the threshold comparison is `>=` (an image with confidence exactly $\tau$ is a candidate);
* ties in confidence within a class are broken by the **lower pool index**;
* ties in the arg-max (two classes with equal top probability) go to the **lower class index**, which is what `torch.argmax` returns;
* the returned indices are sorted ascending, with their labels in the same order.

Practice p04 applies these rules to a literal probability table by hand; p09 implements them as `select_pseudo_labels(probs, threshold, max_per_class)`.

**Balance is a prior.** Capping per class assumes the classes are roughly balanced in the pool.
If you know the pool is imbalanced, caps should follow the expected class proportions, estimated from labelled data, never from test labels.

**Checkpoint 3A.** Probabilities for four pool images, all predicted class 0, have confidences $0.95, 0.97, 0.95, 0.80$ at indices $0..3$. With $\tau=0.9$ and a cap of 2, which indices are kept?

**Checkpoint 3B.** Which tie rule decides the label of an image with probabilities $(0.45, 0.45, 0.10)$?

**Collected answers.** Candidates are indices 0, 1, 2 (index 3 is below $\tau$); ranked by confidence: 1 ($0.97$), then 0 and 2 tied at $0.95$, broken by lower index, so 0; kept: $[0, 1]$. The arg-max tie goes to the lower class index, so label 0 (its confidence is $0.45$).

## 4. Confirmation bias and the noisy student

**Confirmation bias.** A student trained to reproduce its teacher's labels learns the teacher's decision function, mistakes included.
If the student has the same architecture, the same initialization, and sees the pseudo-labelled images exactly as the teacher saw them, it can fit them perfectly and generalize no better than the teacher.
Worse, its confident mistakes are re-admitted in the next round with even higher confidence.
On this unit's data, the recipe of Sections 2–3 without noise changes test accuracy by between about $-0.03$ and $+0.01$ across initialization seeds: the student mostly copies the teacher.

**The noisy student.** Make the student's job *harder* than the teacher's: train it on noisy versions of its inputs.
In this unit the student adds fresh Gaussian noise to every training image at every step,
$$x\leftarrow x+0.3\,\varepsilon,\qquad \varepsilon\sim N(0,I)\ \text{drawn from the global torch generator},$$
while the teacher labels the **clean** pool images.
The student can no longer fit each pseudo-labelled image's exact noise pattern, so it must learn the shape that stays stable under noise, and that function generalizes better than the teacher's.
The idea is the same as Book 1's regularization against `book1:overfitting`: limit what the model can memorize.

**The ablation that matters.** If noise alone (labelled data only, with the same noise) gave the gain, the pool would be irrelevant.
On this unit's data, noise alone moves test accuracy by between about $-0.03$ and $+0.02$ across initialization seeds, while noise plus pseudo-labels gains about $0.04$ to $0.08$; Practice p17 asks you to report this ablation on validation beside your certified result.

**Symptoms of confirmation bias in a trace** (Practice p25): pseudo-label counts that grow fastest for one class; average confidence of admitted images rising toward $1$ while validation accuracy stalls or falls; a class whose validation recall collapses.
Remedies: per-class caps, a higher threshold, re-initialized students, student noise, fewer rounds chosen on validation.

**Checkpoint 4A.** Why does adding noise to the student's inputs, but not the teacher's, help?

**Checkpoint 4B.** A trace shows validation accuracy $0.80 \to 0.83 \to 0.79$ over rounds 0–2 while admitted pseudo-labels per class go from $(100,100,100)$ to $(200,190,70)$. Name the problem and one remedy.

**Collected answers.** The teacher's labels on clean images are its best guesses, but a student fitting noisy copies cannot memorize the teacher's idiosyncratic decision boundary on each exact image, so it learns a smoother, noise-robust function that generalizes better than the teacher. Class imbalance growing in the pseudo-labels (class 2 starved) with confirmation bias; stop at round 1 (chosen on validation), and enforce equal per-class caps (or a higher threshold).

## 5. Cluster-then-label with k-means

A second way to use the pool assumes the **cluster assumption**: images of the same class lie close together in some feature space, and classes are separated by gaps.
Then clustering the pool finds the classes without labels, and the labelled images only need to name the clusters.

**The recipe in this unit.**

1. Train the supervised baseline CNN and **freeze** it.
2. Define the feature map `features(x) = model.conv1(x).flatten(1)`: the 8 first-layer filter responses at each of the $6\times6$ positions, a 288-dimensional vector per image, taken before the ReLU. These are frozen features; nothing is trained further.
3. Run `book1:lloyd-algorithm` (k-means) with $k=3$ on the features of the 30 labelled training images followed by the 600 pool images (630 rows, in that order):
   * initialize centroid $j$ at the mean feature vector of the labelled images of class $j$ (so cluster $j$ starts near class $j$);
   * for exactly 20 iterations: assign every row to its nearest centroid (Euclidean distance, ties to the lower centroid index), then move each centroid to the mean of its assigned rows; a centroid with no rows stays where it is.
4. **Name the clusters by majority vote of the labelled training images** assigned to them; a majority tie goes to the lowest class index.
   A cluster that contains no labelled image takes the label of the nearest *labelled-class centroid* (the step-3 initial centroid of each class) in the frozen feature space.
5. Predict a new image's class as the name of its nearest final centroid.

**Why it can beat the baseline.** The baseline's final linear layer was fitted to 30 images; the k-means centroids are fitted to 630.
In the frozen first-layer feature space the three classes form fairly compact clouds, and centroids averaged over hundreds of noisy images locate those clouds far better than 30 labels can.

**Why it can fail.** If the features are poor (a badly initialized first layer, or features where nuisance factors such as position dominate the distances), the clusters follow the nuisance instead of the classes, and the majority vote cannot fix that.
At the frozen seed this recipe beats the baseline comfortably; with other initialization seeds it sometimes does not, which is exactly why the comparison must be made on validation, and why Practice p18 also compares it with threshold pseudo-labels.

**Checkpoint 5A.** Why initialize the centroids at the labelled class means rather than at random pool images?

**Checkpoint 5B.** A final cluster contains labelled images with classes $(1, 1, 2, 2)$. What label does it get?

**Collected answers.** It ties cluster $j$ to class $j$ from the start, makes the result deterministic, and avoids two initial centroids landing in the same class cloud. The majority is tied between classes 1 and 2, so it goes to the lowest class index: label 1.

## 6. Leakage-safe validation and the semi-supervised protocol

Semi-supervised learning adds a new data role, so the protocol must say exactly where each row may go:

| rows | count | may enter training? | labels visible? | used for selection? |
|---|---:|---|---|---|
| labelled training | 30 | yes | yes | no |
| labelled validation | 30 | **no** | yes | **yes** |
| unlabelled pool | 600 | yes, with pseudo-labels or in k-means | **never** | no |
| locked test | 300 | **no** | only inside `final_test_score` | **no** |

**The pool's labels never exist for you.** `capstone_data.unlabelled_rows()` returns images only, and the stored data file does not even contain the pool's labels.
Any diagnostic that needs true labels (pseudo-label accuracy, per-class recall) must be measured on the labelled validation images instead.

**The seam check.** In p17 the seam is the model's `forward` during training: every row that reaches it must hash to `"train"` or `"unlabelled"` via `capstone_data.split_of("shapes_ssl", row)`.
In p18 the seam is `features(x)` while k-means is fitted: again only `"train"` and `"unlabelled"` rows.
Because the student's noise is added *inside* the training loop, hash the clean rows before the noise (or check inside `forward` of a wrapper that receives the clean batch and adds the noise itself).

**The test call.** After all rounds and selections, call `final_test_score` exactly once with a `predict_fn` that returns int64 labels of shape `(N,)`.
Its accuracy and bootstrap interval are what you report against the baseline's.

**Checkpoint 6A.** You want the accuracy of your pseudo-labels. On which images can you measure it, and how?

**Checkpoint 6B.** Where must the student's input noise be added so that the seam check still sees stored rows?

**Collected answers.** On the labelled validation images: pseudo-label them with the teacher exactly as you would the pool, and compare with their true labels (without ever training on them). After the seam: hash the clean batch first (or have a wrapper's `forward` hash its clean input and then add the noise before calling the CNN).

## 7. Worked laboratory: a tiny shape trace

The laboratory runs a reduced version of both methods on `shapes_ssl`, using **validation rows only** (no test call): the supervised baseline; one round of noisy-student self-training with $\tau=0.9$ and a cap of 100 per class; the noise-only ablation; and cluster-then-label on frozen first-layer features.
It prints the pseudo-label class counts, validation accuracies, and the k-means cluster names.
Compare the validation accuracies, and remember that with 30 validation images one image is $0.033$.

**Checkpoint 7A.** Why does the laboratory print the pseudo-label class counts?

**Checkpoint 7B.** Which printed number is the noise-only ablation, and what would it mean if it matched the self-training number?

**Collected answers.** To check class balance (the cap should keep counts at most 100 each) and to spot a teacher that floods one class. The validation accuracy of the CNN trained on labelled images with student noise but no pool; if it matched self-training, the gain would come from the noise, not from the pseudo-labels.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

xl, yl = capstone_data.train_rows("shapes_ssl")
xv, yv = capstone_data.val_rows("shapes_ssl")
pool = capstone_data.unlabelled_rows()
print("labelled", tuple(xl.shape), "validation", tuple(xv.shape), "pool", tuple(pool.shape))

def train_cnn(x, y, noise_std=0.0, steps=150):
    torch.manual_seed(SEED)
    model = capstone_data.ShapeCNN()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(steps):
        xb = x + noise_std * torch.randn_like(x) if noise_std else x
        optimizer.zero_grad(set_to_none=True)
        F.cross_entropy(model(xb), y).backward()
        optimizer.step()
    return model.eval()

def val_accuracy(model):
    with torch.no_grad():
        return (model(xv).argmax(dim=1) == yv).double().mean().item()

def select(probs, threshold, cap):
    conf, label = probs.max(dim=1)
    keep = []
    for k in range(probs.shape[1]):
        cand = torch.nonzero((label == k) & (conf >= threshold)).flatten().tolist()
        cand.sort(key=lambda i: (-conf[i].item(), i))
        keep += cand[:cap]
    keep = torch.tensor(sorted(keep), dtype=torch.int64)
    return keep, label[keep]

baseline = train_cnn(xl, yl)
with torch.no_grad():
    probs = F.softmax(baseline(pool), dim=1)
idx, pseudo = select(probs, 0.9, 100)
student = train_cnn(torch.cat([xl, pool[idx]]), torch.cat([yl, pseudo]), noise_std=0.3)
noise_only = train_cnn(xl, yl, noise_std=0.3)
print("pseudo-label class counts:", torch.bincount(pseudo, minlength=3).tolist())
print(f"validation accuracy: baseline {val_accuracy(baseline):.3f} | one noisy-student round {val_accuracy(student):.3f} "
      f"| noise-only ablation {val_accuracy(noise_only):.3f}")

In [ ]:
def features(x):
    with torch.no_grad():
        return baseline.conv1(x).flatten(1)             # frozen first-layer responses, (N, 288)

Z = torch.cat([features(xl), features(pool)])            # labelled first, then pool
init = torch.stack([features(xl)[yl == k].mean(dim=0) for k in range(3)])
centroids = init.clone()
for _ in range(20):
    assign = torch.cdist(Z, centroids).argmin(dim=1)
    centroids = torch.stack([Z[assign == j].mean(dim=0) if (assign == j).any() else centroids[j] for j in range(3)])

lab_assign = torch.cdist(features(xl), centroids).argmin(dim=1)
names = []
for j in range(3):
    members = yl[lab_assign == j]
    if len(members):
        names.append(int(torch.bincount(members, minlength=3).argmax()))   # ties -> lowest class
    else:
        names.append(int(torch.cdist(centroids[j:j + 1], init).argmin()))  # nearest labelled-class centroid
names = torch.tensor(names)
pred_val = names[torch.cdist(features(xv), centroids).argmin(dim=1)]
print("cluster names:", names.tolist(), "| cluster sizes:", torch.bincount(assign, minlength=3).tolist())
print(f"cluster-then-label validation accuracy {(pred_val == yv).double().mean().item():.3f}")

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Validation images used as training or pseudo-labelled data.* Fix: validation is labelled, held out, and used only for selection.
2. *Measuring pseudo-label accuracy with pool labels.* Fix: there are none; measure on validation.
3. *`>` instead of `>=` at the threshold, or no tie rule.* Fix: the deterministic rules of Section 3.
4. *No per-class cap.* Fix: caps per predicted class; watch the counts.
5. *Student identical to teacher (no noise, same data).* Fix: noisy student; re-initialized students; report the noise-only ablation.
6. *Noise added before the seam check.* Fix: hash clean rows, then add noise.
7. *k-means with random initialization and no stated tie rules.* Fix: labelled-class-mean initialization, 20 iterations, majority with lowest-index ties, nearest-labelled-centroid rule for unlabelled clusters.
8. *Choosing the method on the test set.* Fix: validation, then one `final_test_score` call.

**Exam connections.** p04 applies a threshold and per-class cap to a literal table; p09 implements `select_pseudo_labels`; p17 certifies that noisy-student self-training beats the fixed labelled-only baseline by at least $0.03$ test accuracy with selection on validation; p18 certifies cluster-then-label against the same baseline and compares it with threshold pseudo-labels; p25 diagnoses a pseudo-labeling trace that goes wrong and chooses a remedy.

**Checkpoint 8A.** Which pitfall does a seam check inside the model's `forward` catch?

**Checkpoint 8B.** Which two practices certify a margin over the same fixed baseline?

**Collected answers.** Validation or test images reaching training (pitfall 1, and test leakage), as long as the check sees clean rows. p17 and p18.

## 9. Forward-only deeper topic: consistency regularization and label propagation

Forward-only deeper topic.
**Consistency regularization** trains the model to give the same prediction for two perturbations of the same unlabelled image, without hard pseudo-labels; **FixMatch** combines this with confidence thresholds: a weakly perturbed view produces a pseudo-label (if confident) that supervises a strongly perturbed view.
**MixMatch** and **mean-teacher** methods average predictions over augmentations or over a moving average of model weights to make pseudo-labels less noisy.
**Label propagation** builds a nearest-neighbour graph over labelled and unlabelled points and spreads labels along its edges, an explicit form of the cluster assumption.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-024.

This final section is forward-only and not assessed.

**Checkpoint 9A.** In FixMatch, which view produces the pseudo-label and which view is trained on it?

**Checkpoint 9B.** Which assumption does label propagation make explicit?

**Collected answers.** The weakly perturbed view produces it (if confident); the strongly perturbed view is trained to predict it. The cluster (manifold) assumption: nearby points in the graph share labels.